# Lab 10 — VQE: Escaneo de Energía

Exploramos cómo varía la energía VQE en función de los parámetros del ansatz.
Visualizamos el paisaje de energía 1D y 2D, e identificamos el mínimo global vs mínimos locales.

**Objetivo**: entender cuándo VQE converge al estado fundamental y cuándo queda atrapado.

## 1. Hamiltoniano y ansatz

Usamos el modelo de Ising transversal 1D con 2 spins:

$$H = -J Z_0 Z_1 - h(X_0 + X_1)$$

con J=1, h=0.5. El estado fundamental cambia de carácter a medida que h/J varía.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize, differential_evolution
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives import StatevectorEstimator

J, h = 1.0, 0.5
H_ising = SparsePauliOp.from_list([
    ('ZZ', -J),
    ('XI', -h),
    ('IX', -h),
])

# Energía exacta
E_exact = np.linalg.eigvalsh(H_ising.to_matrix())
print(f'J={J}, h={h}')
print(f'Autovalores exactos: {E_exact}')
print(f'E₀ = {E_exact[0]:.6f}')

## 2. Escaneo 1D: rotación de un qubit

Fijamos θ₁=0 y escaneamos θ₀ ∈ [-π, π].
Esto muestra cómo la energía depende de un solo parámetro.

In [ ]:
estimator = StatevectorEstimator()

# Ansatz simple: Ry(θ₀) ⊗ Ry(θ₁) + CNOT
theta = ParameterVector('θ', 2)
ansatz = QuantumCircuit(2)
ansatz.ry(theta[0], 0)
ansatz.ry(theta[1], 1)
ansatz.cx(0, 1)
ansatz.ry(theta[0], 0)
ansatz.ry(theta[1], 1)

# Escaneo 1D
t0_values = np.linspace(-np.pi, np.pi, 100)
energies_1d = []
for t0 in t0_values:
    job = estimator.run([(ansatz, H_ising, [t0, 0.0])])
    energies_1d.append(float(job.result()[0].data.evs))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(t0_values / np.pi, energies_1d, 'b-', linewidth=2, label='⟨H⟩(θ₀, θ₁=0)')
ax.axhline(E_exact[0], color='red', linestyle='--', label=f'E₀ = {E_exact[0]:.3f}')
ax.set_xlabel('θ₀ / π', fontsize=12)
ax.set_ylabel('⟨H⟩', fontsize=12)
ax.set_title('Escaneo 1D de energía VQE', fontsize=13)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Mínimo 1D: {min(energies_1d):.6f} (E₀={E_exact[0]:.6f})')

## 3. Escaneo 2D: paisaje completo

El paisaje 2D revela la topología del espacio de optimización:
- ¿Hay un único mínimo global?
- ¿Cuántos mínimos locales existen?
- ¿El gradiente es informativo (no hay barren plateaus)?

In [ ]:
n = 35
t0s = np.linspace(-np.pi, np.pi, n)
t1s = np.linspace(-np.pi, np.pi, n)
Z = np.zeros((n, n))

for i, t0 in enumerate(t0s):
    for j, t1 in enumerate(t1s):
        job = estimator.run([(ansatz, H_ising, [t0, t1])])
        Z[i, j] = float(job.result()[0].data.evs)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Mapa de calor
c = axes[0].contourf(t0s/np.pi, t1s/np.pi, Z.T, levels=25, cmap='plasma')
plt.colorbar(c, ax=axes[0], label='⟨H⟩')
min_idx = np.unravel_index(Z.argmin(), Z.shape)
axes[0].plot(t0s[min_idx[0]]/np.pi, t1s[min_idx[1]]/np.pi, 'r*', markersize=15, label='Mínimo')
axes[0].set_xlabel('θ₀ / π'); axes[0].set_ylabel('θ₁ / π')
axes[0].set_title('Paisaje de energía 2D')
axes[0].legend()

# Superficie 3D
from mpl_toolkits.mplot3d import Axes3D
T0, T1 = np.meshgrid(t0s/np.pi, t1s/np.pi)
axes[1] = fig.add_subplot(122, projection='3d')
axes[1].plot_surface(T0, T1, Z.T, cmap='plasma', alpha=0.8)
axes[1].set_xlabel('θ₀/π'); axes[1].set_ylabel('θ₁/π'); axes[1].set_zlabel('⟨H⟩')
axes[1].set_title('Superficie 3D')

plt.suptitle(f'Paisaje VQE — Ising J={J}, h={h}', fontsize=13)
plt.tight_layout()
plt.show()
print(f'Mínimo encontrado en grid: {Z.min():.6f} (E₀={E_exact[0]:.6f})')

## 4. Comparación: gradiente descendente vs evolución diferencial

Los optimizadores locales (COBYLA, BFGS) pueden quedar atrapados en mínimos locales. Los globales (evolución diferencial) exploran todo el espacio pero son más lentos.

In [ ]:
results = {}

def cost(params):
    return float(estimator.run([(ansatz, H_ising, params)]).result()[0].data.evs)

# COBYLA (local) desde punto aleatorio
np.random.seed(99)
x0 = np.random.uniform(-np.pi, np.pi, 2)
res_cobyla = minimize(cost, x0, method='COBYLA', options={'maxiter': 300})
results['COBYLA'] = res_cobyla.fun

# L-BFGS-B (gradiente local)
res_bfgs = minimize(cost, x0, method='L-BFGS-B',
                    options={'maxiter': 300},
                    bounds=[(-np.pi, np.pi)] * 2)
results['L-BFGS-B'] = res_bfgs.fun

# Evolución diferencial (global)
res_de = differential_evolution(cost, bounds=[(-np.pi, np.pi)] * 2,
                                maxiter=100, seed=42, tol=1e-6)
results['Evol. Diferencial'] = res_de.fun

print(f'E₀ exacto:              {E_exact[0]:.6f}')
for name, val in results.items():
    print(f'{name:<22}: {val:.6f}  (error={abs(val-E_exact[0]):.2e})')

## 5. Conclusiones

**Lo que aprendemos del escaneo de energía**:
- El paisaje puede tener múltiples mínimos locales: el optimizador local depende del punto de inicio
- La evolución diferencial encuentra el óptimo global pero requiere más evaluaciones
- Para Hamiltonianos más complejos (más qubits), el paisaje puede exhibir **barren plateaus** donde el gradiente ≈ 0 en todo el espacio
- La elección del ansatz es crucial: un ansatz con pocas capas puede no alcanzar el estado fundamental

In [ ]:
# Resumen de evaluaciones de función
print('Optimizador        | Evaluaciones | Energía final | Error')
print('-' * 60)
print(f'COBYLA             | {res_cobyla.nfev:<12} | {res_cobyla.fun:.6f}    | {abs(res_cobyla.fun-E_exact[0]):.2e}')
print(f'L-BFGS-B           | {res_bfgs.nfev:<12} | {res_bfgs.fun:.6f}    | {abs(res_bfgs.fun-E_exact[0]):.2e}')
print(f'Evol. Diferencial  | {res_de.nfev:<12} | {res_de.fun:.6f}    | {abs(res_de.fun-E_exact[0]):.2e}')